In [1]:
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment
import torch
import torchaudio

In [2]:
SAMPLE_WAV_FILE = "./sample_audio.wav"

In [3]:
audio, sample_rate = librosa.load(SAMPLE_WAV_FILE)
clip = AudioSegment.from_wav(SAMPLE_WAV_FILE)

bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()
labels = bundle.get_labels()

/Users/jwen/Documents/Semester 5/Speech Recog/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#submenu utk menu 2
class GreedyCTCDecoder(torch.nn.Module):
    def __init__(self, labels):
        super().__init__()
        self.labels = labels
        self.blank = 0

    def forward(self, emission: torch.Tensor) -> str:
        # Pilih label dengan skor tertinggi pada setiap langkah waktu.
        indices = torch.argmax(emission, dim=-1)

        # Gabungkan label sama yang muncul berurutan.
        indices = torch.unique_consecutive(indices, dim=-1)

        # Hapus blank, lalu ubah indeks menjadi karakter.
        indices = [i for i in indices if i != self.blank]
        return "".join([self.labels[i] for i in indices])


decoder = GreedyCTCDecoder(labels=labels)

In [5]:
#get audio file
def get_filename():
    filename = input("New filename (.wav): ")

    while not filename.endswith(".wav"):
        filename = input("Filename must end with .wav: ")

    return filename

In [6]:
#MENU 1A
def pitch_shift():
    while True:
        try:
            value = float(input("Pitch shift [-5 to 5]: "))
            if -5 <= value <= 5:
                break
        except ValueError:
            pass

        print("Invalid value.")

    filename = get_filename()
    result = librosa.effects.pitch_shift(
        audio, sr=sample_rate, n_steps=value
    )
    sf.write(filename, result, sample_rate)
    print("Saved:", filename)

In [7]:
#MENU 1B
def white_noise(frames, factor):
    noise = np.random.normal(0, frames.std(), frames.size)
    return frames + noise * factor


def add_white_noise():
    while True:
        try:
            value = float(input("Noise level [0.1 to 0.5]: "))
            if 0.1 <= value <= 0.5:
                break
        except ValueError:
            pass

        print("Invalid value.")

    filename = get_filename()
    result = white_noise(audio, value)
    sf.write(filename, result, sample_rate)
    print("Saved:", filename)

In [8]:
#MENU 1C
def repeat_clip():
    while True:
        try:
            value = int(input("Repeat count [greater than 0]: "))
            if value > 0:
                break
        except ValueError:
            pass

        print("Invalid value.")

    filename = get_filename()
    result = clip * value
    result.export(filename, format="wav")
    print("Saved:", filename)

In [9]:
#MENU 1D
def fade_in():
    while True:
        try:
            value = int(input("Fade-in duration in ms [greater than 0]: "))
            if value > 0:
                break
        except ValueError:
            pass

        print("Invalid value.")

    filename = get_filename()
    result = clip.fade_in(value)
    result.export(filename, format="wav")
    print("Saved:", filename)

In [10]:
#MENU 2
def predict_text():
    waveform, sr = torchaudio.load(SAMPLE_WAV_FILE)

    if sr != bundle.sample_rate:
        waveform = torchaudio.functional.resample(
            waveform, sr, bundle.sample_rate
        )

    with torch.inference_mode():
        emission, _ = model(waveform)

    transcript = decoder(emission[0])
    print(transcript.replace("|", " "))

In [11]:
#MENU 1
def augmentation_menu():
    while True:
        print()
        print("1. Pitch shift")
        print("2. Add white noise")
        print("3. Repeat clip")
        print("4. Insert fade in")
        print("5. Back to main menu")
        choice = input("Choose: ")

        if choice == "1":
            pitch_shift()
        elif choice == "2":
            add_white_noise()
        elif choice == "3":
            repeat_clip()
        elif choice == "4":
            fade_in()
        elif choice == "5":
            break
        else:
            print("Invalid choice.")

In [12]:
def main_menu():
    while True:
        print()
        print("1. Audio Augmentation")
        print("2. Predict Text")
        print("3. Exit")
        choice = input("Choose: ")

        if choice == "1":
            augmentation_menu()
        elif choice == "2":
            predict_text()
        elif choice == "3":
            break
        else:
            print("Invalid choice.")

main_menu()


1. Audio Augmentation
2. Predict Text
3. Exit

1. Pitch shift
2. Add white noise
3. Repeat clip
4. Insert fade in
5. Back to main menu
Saved: new.wav

1. Pitch shift
2. Add white noise
3. Repeat clip
4. Insert fade in
5. Back to main menu
Saved: 2.wav

1. Pitch shift
2. Add white noise
3. Repeat clip
4. Insert fade in
5. Back to main menu
Saved: 3.wav

1. Pitch shift
2. Add white noise
3. Repeat clip
4. Insert fade in
5. Back to main menu
Saved: 4.wav

1. Pitch shift
2. Add white noise
3. Repeat clip
4. Insert fade in
5. Back to main menu

1. Audio Augmentation
2. Predict Text
3. Exit


/Users/jwen/Documents/Semester 5/Speech Recog/.venv/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


THE PUBLIC ARE ENTREATED TO BEAR IN MIND THAT THIRTEEN YEARS HAVE PASSED SINCE IT WAS FINISHED MANY MORE SINCE IT WAS BEGUN AND THAT DURING THAT PERIOD PLACES MANNERS BOOKS AND OPINIONS HAVE UNDERGONE CONSIDERABLE CHANGES 

1. Audio Augmentation
2. Predict Text
3. Exit


In [13]:
#augmentation
# 1. Menaikkan volume sebesar 10 db. Jika ingin menurunkan, tinggal dikurang
volume_audio = clip + 10
volume_audio.export("volume.wav", format="wav")

# 2. Membuat audio 2x lebih cepat. Jika rate=0.5, audio akan lebih lambat sebanyak 2x
speed_audio = librosa.effects.time_stretch(audio, rate=2.0)
sf.write("speed.wav", speed_audio, sample_rate)

# 3. Reverse audio.
reverse_audio = clip.reverse()
reverse_audio.export("reverse.wav", format="wav")

# 4. Mengalikan amplitudo dengan faktor pengali secara acak (random gain)
# Faktor < 1 mengecilkan amplitudo, dan faktor > 1 membesarkan amplitudo.
gain = np.random.uniform(0.5, 1.5)
gain_audio = audio * gain
sf.write("random_gain.wav", gain_audio, sample_rate)